# GTU Mimari Lejant - Colab Training

Bu notebook Roboflow YOLOv11 instance segmentation export'u ile ilk YOLO segmentation baseline modelini egitir.

Colab ayari: `Runtime -> Change runtime type -> GPU`. GPU onceligi: A100 > L4 > T4.

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## 1. Repo'yu Klonla

In [ ]:
%cd /content
!rm -rf lejanter_doga_vlm_codex
!git clone https://github.com/doganalci/lejanter_doga_vlm_codex.git
%cd /content/lejanter_doga_vlm_codex
!pip install -q -r requirements.txt

## 2. Roboflow Zip'i Yukle

Roboflow'dan indirdigin `GTU_MIMARI_LEJANT.yolov11.zip` dosyasini sec.

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_name = next(iter(uploaded.keys()))
print('Uploaded:', zip_name)

Alternatif: Dosya Google Drive'daysa bu hucreyi kullanabilirsin. Upload kullandiysan bu hucreyi atla.

```python
from google.colab import drive
drive.mount('/content/drive')
zip_name = '/content/drive/MyDrive/GTU_MIMARI_LEJANT.yolov11.zip'
```

## 3. Dataset'i Ac ve Split Hazirla

In [ ]:
from pathlib import Path
import shutil
import yaml

dataset_root = Path('data/roboflow/gtu-mimari-lejant')
if dataset_root.exists():
    shutil.rmtree(dataset_root)
dataset_root.mkdir(parents=True, exist_ok=True)

!unzip -q "{zip_name}" -d data/roboflow/gtu-mimari-lejant

# Roboflow export bazen sadece train split'i ile gelir. Valid/test yoksa olustur.
if not Path('data/roboflow/gtu-mimari-lejant/valid/images').exists():
    !python scripts/split_yolo_dataset.py --root data/roboflow/gtu-mimari-lejant --valid 0.1 --test 0.1
else:
    print('valid/test split already exists')

# Colab icin path'i mutlak hale getiriyoruz; Ultralytics relative path'i farkli yorumlayabiliyor.
cfg = yaml.safe_load(Path('configs/elements_dataset.yaml').read_text())
cfg['path'] = str(dataset_root.resolve())
Path('configs/elements_colab.yaml').write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
print(Path('configs/elements_colab.yaml').read_text())

In [ ]:
!find data/roboflow/gtu-mimari-lejant -maxdepth 3 -type f | awk -F/ '{print $(NF-2) "/" $(NF-1)}' | sort | uniq -c
!sed -n '1,80p' configs/elements_colab.yaml
!test -d data/roboflow/gtu-mimari-lejant/train/images && test -d data/roboflow/gtu-mimari-lejant/valid/images && test -d data/roboflow/gtu-mimari-lejant/test/images && echo 'Dataset folders OK'

## 4. Ilk Baseline Egitimi

T4 GPU'da bellek sorunu olursa `yolo11s-seg.pt` yerine `yolo11n-seg.pt`, `imgsz=768` kullan.

In [ ]:
!python scripts/train_yolo.py \
  --data configs/elements_colab.yaml \
  --model yolo11s-seg.pt \
  --task segment \
  --name elements-seg-v1 \
  --epochs 100 \
  --imgsz 1024 \
  --batch -1 \
  --device 0

## 5. Test Set Inference

In [ ]:
!rm -rf outputs/runs/infer-elements-test
!python scripts/infer_yolo.py \
  --weights outputs/runs/elements-seg-v1/weights/best.pt \
  --source data/roboflow/gtu-mimari-lejant/test/images \
  --task segment \
  --out outputs/reports/elements_test.json \
  --name infer-elements-test \
  --save-visuals \
  --device 0

!ls -R outputs/runs/infer-elements-test | sed -n '1,160p'

## 6. Sonuclari Goster

In [ ]:
from IPython.display import Image, display
from pathlib import Path

pred_dir = Path('outputs/runs/infer-elements-test')
if not pred_dir.exists():
    raise FileNotFoundError(f'Prediction folder not found: {pred_dir}')
for image_path in list(pred_dir.glob('*'))[:5]:
    if image_path.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
        display(Image(filename=str(image_path)))

## 7. Kendi Gorselini Dene

Egitim bittikten sonra yeni bir cephe fotografi yukleyip `best.pt` ile sonucu gorebilirsin. Bir veya birden fazla `.jpg/.png` dosyasi secilebilir.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

custom_dir = Path('data/raw/custom_uploads')
if custom_dir.exists():
    shutil.rmtree(custom_dir)
custom_dir.mkdir(parents=True, exist_ok=True)

custom_uploaded = files.upload()
for name, content in custom_uploaded.items():
    (custom_dir / name).write_bytes(content)

print('Uploaded custom images:')
for path in custom_dir.iterdir():
    print('-', path)

In [ ]:
!rm -rf outputs/runs/infer-custom-upload
!python scripts/infer_yolo.py \
  --weights outputs/runs/elements-seg-v1/weights/best.pt \
  --source data/raw/custom_uploads \
  --task segment \
  --out outputs/reports/custom_uploads.json \
  --name infer-custom-upload \
  --save-visuals \
  --device 0

!ls -R outputs/runs/infer-custom-upload | sed -n '1,160p'

In [ ]:
from IPython.display import Image, display
from pathlib import Path
import json

pred_dir = Path('outputs/runs/infer-custom-upload')
if not pred_dir.exists():
    raise FileNotFoundError(f'Prediction folder not found: {pred_dir}')
for image_path in sorted(pred_dir.glob('*')):
    if image_path.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
        display(Image(filename=str(image_path)))

custom_json = Path('outputs/reports/custom_uploads.json')
if custom_json.exists():
    data = json.loads(custom_json.read_text())
    print(json.dumps(data[:1], ensure_ascii=False, indent=2)[:3000])

In [ ]:
!zip -r custom_upload_predictions.zip outputs/runs/infer-custom-upload outputs/reports/custom_uploads.json
files.download('custom_upload_predictions.zip')

## 8. Model ve Raporlari Indir

In [ ]:
!zip -r gtu_elements_seg_v1_results.zip outputs/runs/elements-seg-v1 outputs/runs/infer-elements-test outputs/reports/elements_test.json
files.download('gtu_elements_seg_v1_results.zip')